# Text Summarization using LSTM

This notebook demonstrates how to build an LSTM-based encoder-decoder model for abstractive text summarization using TensorFlow/Keras.

In [1]:
import numpy as np
import pandas as pd
import re
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

2025-08-05 08:05:15.251041: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754381115.446882      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754381115.500331      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
df = pd.read_csv('/kaggle/input/news-summary/news_summary.csv', encoding='ISO-8859-1')
df = df[['text', 'ctext']]
df.rename(columns={'ctext': 'text', 'text': 'summary'}, inplace=True)
df.dropna(inplace=True)

In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub('"','', text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['clean_text'] = df['text'].apply(clean_text)
df['clean_summary'] = df['summary'].apply(clean_text)
df['clean_summary'] = df['clean_summary'].apply(lambda x : 'sostok ' + x + ' eostok')

In [4]:
max_text_len = 300
max_summary_len = 40

x_tokenizer = Tokenizer()
x_tokenizer.fit_on_texts(df['clean_text'])
x_tr_seq = x_tokenizer.texts_to_sequences(df['clean_text'])
x_padded = pad_sequences(x_tr_seq, maxlen=max_text_len, padding='post')

y_tokenizer = Tokenizer()
y_tokenizer.fit_on_texts(df['clean_summary'])
y_tr_seq = y_tokenizer.texts_to_sequences(df['clean_summary'])
y_padded = pad_sequences(y_tr_seq, maxlen=max_summary_len, padding='post')

x_voc = len(x_tokenizer.word_index) + 1
y_voc = len(y_tokenizer.word_index) + 1

In [5]:
x_train, x_val, y_train, y_val = train_test_split(x_padded, y_padded, test_size=0.1, random_state=0)

In [6]:
latent_dim = 300
embedding_dim = 100

encoder_inputs = Input(shape=(max_text_len,))
enc_emb = Embedding(x_voc, embedding_dim, trainable=True)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_output, state_h, state_c = encoder_lstm(enc_emb)

decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(y_voc, embedding_dim, trainable=True)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])
decoder_dense = Dense(y_voc, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')
model.summary()

I0000 00:00:1754381130.609116      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 300, 100)  │  4,762,400 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │  1,846,600 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 300,      │    481,200 │ embedding[0][0]   │
│                     │ 300), (None,      │            │                   │
│                     │ 300), (None,      │            │                   │
│                     │ 300)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    481,200 │ embedding_1[0][0… │
│                     │ 300), (None,      │            │ lstm[0][1],       │
│                     │ 300), (None,      │            │ lstm[0][2]        │
│                     │ 300)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  5,558,266 │ lstm_1[0][0]      │
│                     │ 18466)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,129,666 (50.09 MB)

 Trainable params: 13,129,666 (50.09 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
y_train_decoder = np.expand_dims(y_train[:,1:], -1)
y_val_decoder = np.expand_dims(y_val[:,1:], -1)

In [8]:
model.fit([x_train, y_train[:, :-1]],
          y_train_decoder,
          epochs=200,
          batch_size=128,
          validation_data=([x_val, y_val[:, :-1]], y_val_decoder))

Epoch 1/200


I0000 00:00:1754381136.024970      95 cuda_dnn.cc:529] Loaded cuDNN version 90300


31/31 ━━━━━━━━━━━━━━━━━━━━ 9s 153ms/step - loss: 9.5064 - val_loss: 7.2632
Epoch 2/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 7.2344 - val_loss: 7.1701
Epoch 3/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - loss: 7.1342 - val_loss: 7.1605
Epoch 4/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - loss: 7.1193 - val_loss: 7.1606
Epoch 5/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - loss: 7.1083 - val_loss: 7.1610
Epoch 6/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - loss: 7.1141 - val_loss: 7.1557
Epoch 7/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - loss: 7.1010 - val_loss: 7.1643
Epoch 8/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 7.0928 - val_loss: 7.1639
Epoch 9/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 7.0764 - val_loss: 7.1619
Epoch 10/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - loss: 7.0755 - val_loss: 7.1713
Epoch 11/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 137ms/step - loss: 7.0846 - val_loss: 7.1535
Epoch 12/200
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step

In [9]:
encoder_model_inf = Model(inputs=encoder_inputs, outputs=[encoder_output, state_h, state_c])

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c])
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model_inf = Model([decoder_inputs] + [decoder_state_input_h, decoder_state_input_c],
                          [decoder_outputs2] + [state_h2, state_c2])

reverse_target_word_index = y_tokenizer.index_word
reverse_source_word_index = x_tokenizer.index_word
target_word_index = y_tokenizer.word_index

In [10]:
def decode_sequence(input_seq):
    e_out, e_h, e_c = encoder_model_inf.predict(input_seq)
    target_seq = np.zeros((1,1))
    target_seq[0, 0] = target_word_index['sostok']
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model_inf.predict([target_seq] + [e_h, e_c])
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_token = reverse_target_word_index.get(sampled_token_index, '')
        if sampled_token == 'eostok' or len(decoded_sentence.split()) >= max_summary_len-1:
            stop_condition = True
        else:
            decoded_sentence += ' ' + sampled_token
        target_seq = np.zeros((1,1))
        target_seq[0, 0] = sampled_token_index
        e_h, e_c = h, c
    return decoded_sentence

In [11]:
print("Article:", df['text'].iloc[0])
print("Original Summary:", df['summary'].iloc[0])
print("Predicted Summary:", decode_sequence(x_padded[0].reshape(1, max_text_len)))

Article: The Daman and Diu administration on Wednesday withdrew a circular that asked women staff to tie rakhis on male colleagues after the order triggered a backlash from employees and was ripped apart on social media.The union territory?s administration was forced to retreat within 24 hours of issuing the circular that made it compulsory for its staff to celebrate Rakshabandhan at workplace.?It has been decided to celebrate the festival of Rakshabandhan on August 7. In this connection, all offices/ departments shall remain open and celebrate the festival collectively at a suitable time wherein all the lady staff shall tie rakhis to their colleagues,? the order, issued on August 1 by Gurpreet Singh, deputy secretary (personnel), had said.To ensure that no one skipped office, an attendance report was to be sent to the government the next evening.The two notifications ? one mandating the celebration of Rakshabandhan (left) and the other withdrawing the mandate (right) ? were issued by 